In [2]:
%load_ext autoreload
%autoreload 2

In [3]:

import logging
import os
from crmprtd import setup_logging


setup_logging(None, '/workspaces/crmprtd/fern/all_stations_insert/fern_all_station_log.log', 'tongli1997@uvic.ca', 'DEBUG', 'fern')
log = logging.getLogger(__name__)


import sqlalchemy as sa
from sqlalchemy.orm import Session
from pycds import Variable
from crmprtd.infer import infer
import pickle
from crmprtd.more_itertools import tap, log_progress
from crmprtd.align import align
from crmprtd.insert import insert
from pprint import pprint
from crmprtd.constants import InsertStrategy

print(os.getcwd())
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

# result = session.execute(sa.text("SELECT * FROM crmp.meta_network"))
# test = result.fetchall()

# print(test)

results2 = session.query(Variable).filter(Variable.network_id == 11)
test2 = results2.all()
# pprint(test2)


output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


# file_name = 'updated_rows_with_unit_output_Barren.pickle'
# file_name = 'full_record_station1-2_updated_rows_all_stations.pkl'
file_name = 'updated_all_rows_all_stations.pkl'
file_path = os.path.join(output_folder, file_name)

# Load the pickle file
with open(file_path, 'rb') as f:
    rows = pickle.load(f)

# pprint(rows)
rows_insert = rows

is_diagnostic = False
insert_strategy = InsertStrategy.BULK
bulk_chunk_size=1000
sample_size=50
network = '_test'


infer(session, rows_insert, diagnostic = is_diagnostic)

# Filter the observations by time period, then align them.
log.info("Align + filter: start")
observations = list(
    # Note: filter(None, <collection>) removes falsy values from <collection>,
    # in this case possible None values returned by align.
    filter(
        None,
        tap(
            log_progress(
                (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 5000),
                "align progress: {count}",
                log.info,
            ),
            (
                align(session, row, is_diagnostic)
                for row in rows_insert
                # if start_date <= row.time <= end_date
            ),
        ),
    )
)
log.info("Align + filter: done")

log.info(f"Count of observations to process: {len(observations)}")
if is_diagnostic:
    for obs in observations:
        log.info(obs)
else:

    log.info("Insert: start")
    results = insert(
        session,
        observations,
        strategy=insert_strategy,
        bulk_chunk_size=bulk_chunk_size,
        sample_size=sample_size,
    )
    log.info("Insert: done")
    log.info("Data insertion results", extra={"results": results, "network": network})

# Add this after setting up logging
log.info("Logging system initialized")


/workspaces/crmprtd/fern/all_stations_insert


EOFError: Ran out of input